In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
!pip install requests

In [ ]:
import getpass
API_KEY = getpass.getpass("sec-api key: ")

In [ ]:
import requests, os
URL = "https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm"
os.makedirs("data/nvda_2026/tables", exist_ok=True)

r = requests.get(URL, headers={"User-Agent": "Your Name your@email.com"}, timeout=120)
print(r.status_code, len(r.text))
open("data/nvda_2026/full_10k.htm", "w", encoding="utf-8").write(r.text)

In [ ]:
%pip install lxml beautifulsoup4

In [1]:
from bs4 import BeautifulSoup
import pandas as pd, io

html = open("data/nvda_2026/full_10k.htm", encoding="utf-8").read()
soup = BeautifulSoup(html, "lxml")

# remove the hidden XBRL header block (it is junk metadata, not readable text)
for tag in soup.find_all("ix:header"):
    tag.decompose()

table_count = 0
for t in soup.find_all("table"):
    try:
        df = pd.read_html(io.StringIO(str(t)))[0]
    except ValueError:
        continue
    df = df.dropna(how="all").dropna(axis=1, how="all")
    if df.shape[0] < 2 or df.shape[1] < 2:
        continue                                  # tiny layout tables stay in the text
    df.to_csv(f"data/nvda_2026/tables/table_{table_count}.csv", index=False)
    t.replace_with(soup.new_string(f"\n[TABLE_{table_count}]\n"))   # marker keeps its place in the text
    table_count += 1

text = soup.get_text("\n")
text = "\n".join(line.strip() for line in text.splitlines() if line.strip())
open("data/nvda_2026/full_text.txt", "w", encoding="utf-8").write(text)
print(len(text), "chars of text;", table_count, "tables saved")

/tmp/ipykernel_382/1929827132.py:5: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


307786 chars of text; 61 tables saved
